In [20]:
import pandas as pd
import numpy as np
import utils_old
import utils as new_utils

# -----------------------------
# CONFIGURATION
# -----------------------------
YEARS = 1
PSD_T0_old = PSD_T0.copy()
PSD_T0_new = PSD_T0.copy()

# Helper: initialize opts properly
def initialize_opts(opts, PSD_df):
    """Initialize a CoralOptions object with all required fields."""
    opts.year = 0
    opts.current_surface_area_df = PSD_df.copy()
    opts.current_population_df = PSD_df.copy()

    # Initialize coral cover dictionary
    opts.current_coral_cover = {
        "Branching": PSD_df["Branching"].sum(),
        "Foliose": PSD_df["Foliose"].sum(),
        "Other": PSD_df["Other"].sum(),
    }

    # Initialize benthic cover if missing (needed by get_available_substrate)
    if not hasattr(opts, "current_benthic_cover"):
        opts.current_benthic_cover = {
            "hard_substrate": 0,
            "dead_coral": 0,
            "CCA": 0,
            "turfing_algae": 0,
        }

    # Initialize yearly tracking
    opts.yearly_total_coral_cover_df = pd.DataFrame(columns=[
        "Year",
        "Branching_Area (%)",
        "Foliose_Area (%)",
        "Other_Area (%)",
        "total_coral_cover (%)",
    ])
    opts.yearly_surface_area_df_list = []
    opts.yearly_population_df_list = []

    return opts


# ============================================================
# STEP 1. INITIALIZE OLD MODEL
# ============================================================
old_opts = utils_old.CoralOptions()
old_opts.bleaching = False
old_opts.cyclones = False
old_opts = initialize_opts(old_opts, PSD_T0_old)
utils_old.opts = old_opts

# Run OLD model
old_result = utils_old.run_yearly_change(PSD_T0_old, YEARS)


# ============================================================
# STEP 2. INITIALIZE NEW MODEL
# ============================================================
new_opts = new_utils.CoralOptions()
new_opts.bleaching = False
new_opts.cyclones = False
new_opts = initialize_opts(new_opts, PSD_T0_new)
new_utils.opts = new_opts

# Run NEW model
new_result = new_utils.run_yearly_change(PSD_T0_new, YEARS)


# ============================================================
# STEP 3. COMPARE OUTPUTS
# ============================================================
comparison_df = pd.concat(
    [
        old_result["total_coral_cover (%)"].rename("Old Model"),
        new_result["total_coral_cover (%)"].rename("New Model"),
    ],
    axis=1
)

print("\n===== YEARLY TOTAL CORAL COVER COMPARISON =====\n")
print(comparison_df)
print("\nDifference per year:\n")
print(comparison_df["New Model"] - comparison_df["Old Model"])


/Users/katyaovsyanikova/Documents/GitHub/Coral-Model/4.1 Myrmidon Happy reef/utils.py:759: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  opts.yearly_total_coral_cover_df = pd.concat([opts.yearly_total_coral_cover_df[:],current_df]).reset_index(drop = True)


KeyError: 'macro_algae'